<a href="https://colab.research.google.com/github/hdpuu6626-wq/TKB/blob/main/VisDrone_Small_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("VisDrone Small Object Detection")
print("Enhancing Small Object Detection in Drone Imagery")

VisDrone Small Object Detection
Enhancing Small Object Detection in Drone Imagery


In [2]:
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.1 MB/s eta 0:00:00


In [5]:

%pip install -q ultralytics pyyaml pandas matplotlib


In [6]:

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/visdrone-small-object-detection"

import os
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Thư mục dự án:", PROJECT_DIR)
print("Đã tạo thư mục:", os.path.isdir(PROJECT_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục dự án: /content/drive/MyDrive/visdrone-small-object-detection
Đã tạo thư mục: True


In [7]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Đang dùng CPU")

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [10]:

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection"
)

folders = [
    "data",
    "configs",
    "scripts",
    "src",
    "src/models",
    "notebooks",
    "runs",
    "results",
    "results/figures",
    "results/tables",
]

for folder in folders:
    path = PROJECT_DIR / folder
    path.mkdir(parents=True, exist_ok=True)
    print("Đã tạo/kiểm tra:", path)

print("\nHoàn thành tạo cấu trúc dự án!")


Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/data
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/configs
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/scripts
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/src
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/src/models
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/notebooks
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/runs
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/results
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/results/figures
Đã tạo/kiểm tra: /content/drive/MyDrive/visdrone-small-object-detection/results/tables

Hoàn thành tạo cấu trúc dự án!


Bước 4 — Tự động tải và giải nén dữ liệu

In [11]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection"
)
DATA_DIR = PROJECT_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục dữ liệu:", DATA_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục dữ liệu: /content/drive/MyDrive/visdrone-small-object-detection/data


In [12]:

%pip install -q gdown


In [13]:

import subprocess
import zipfile
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection/data"
)
DOWNLOAD_DIR = DATA_DIR / "downloads"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": {
        "id": "1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn",
        "name": "VisDrone_train.zip",
    },
    "val": {
        "id": "1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59",
        "name": "VisDrone_val.zip",
    },
}

def download_from_drive(file_id, output_path):
    output_path = Path(output_path)

    if output_path.exists() and output_path.stat().st_size > 0:
        print("Đã có file, bỏ qua tải:", output_path.name)
        return

    url = f"https://drive.google.com/uc?id={file_id}"

    result = subprocess.run(
        [
            "gdown",
            "--fuzzy",
            url,
            "-O",
            str(output_path),
        ],
        text=True,
        capture_output=True,
    )

    if result.returncode != 0 or not output_path.exists():
        raise RuntimeError(
            f"Không tải được {output_path.name}.\n"
            f"Chi tiết: {result.stderr}\n"
            "Hãy kiểm tra quyền chia sẻ hoặc tải thủ công từ link chính thức."
        )

    if output_path.stat().st_size == 0:
        raise RuntimeError(f"File tải xuống rỗng: {output_path}")

    print("Tải xong:", output_path.name)
    print("Dung lượng (MB):", round(output_path.stat().st_size / 1024**2, 2))

for item in FILES.values():
    download_from_drive(
        item["id"],
        DOWNLOAD_DIR / item["name"],
    )


Tải xong: VisDrone_train.zip
Dung lượng (MB): 1478.08
Tải xong: VisDrone_val.zip
Dung lượng (MB): 77.86


In [14]:

for item in FILES.values():
    archive = DOWNLOAD_DIR / item["name"]

    if not zipfile.is_zipfile(archive):
        print(
            f"CẢNH BÁO: {archive.name} không phải ZIP hợp lệ. "
            "Có thể Google Drive đã trả về trang xác nhận hoặc lỗi tải."
        )
        continue

    # Mỗi bộ dữ liệu được giải nén vào một thư mục riêng.
    destination = DATA_DIR / archive.stem
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(destination)

    print("Đã giải nén:", archive.name)
    print("Vào thư mục:", destination)

print("\nHoàn tất bước tải và giải nén.")


Đã giải nén: VisDrone_train.zip
Vào thư mục: /content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_train
Đã giải nén: VisDrone_val.zip
Vào thư mục: /content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_val

Hoàn tất bước tải và giải nén.


In [15]:

from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

for split in ["train", "val"]:
    root = DATA_DIR / f"VisDrone_{split}"

    if not root.exists():
        print(f"{split}: chưa tìm thấy {root}")
        continue

    images = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    annotations = list(root.rglob("*.txt"))

    print(f"\n--- {split.upper()} ---")
    print("Thư mục:", root)
    print("Số ảnh:", len(images))
    print("Số file TXT:", len(annotations))
    print("Một vài ảnh:", [p.name for p in images[:3]])



--- TRAIN ---
Thư mục: /content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_train
Số ảnh: 6471
Số file TXT: 6471
Một vài ảnh: ['0000002_00005_d_0000014.jpg', '0000002_00448_d_0000015.jpg', '0000003_00231_d_0000016.jpg']

--- VAL ---
Thư mục: /content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_val
Số ảnh: 548
Số file TXT: 548
Một vài ảnh: ['0000001_02999_d_0000005.jpg', '0000001_03499_d_0000006.jpg', '0000001_03999_d_0000007.jpg']


Bước 5: Chuyển nhãn VisDrone-DET sang định dạng YOLO và tạo tệp cấu hình dữ liệu.


In [16]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DATA_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection/data"
)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_DIR}. "
        "Hãy kiểm tra Bước 3 và Bước 4 trước."
    )

print("Các thư mục và tệp trong data:")
for p in sorted(DATA_DIR.iterdir()):
    print(("📁 " if p.is_dir() else "📄 ") + p.name)

print("\nCác thư mục annotation tìm thấy:")
for p in DATA_DIR.rglob("annotations"):
    if p.is_dir():
        print(p)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Các thư mục và tệp trong data:
📁 VisDrone_train
📁 VisDrone_val
📁 downloads

Các thư mục annotation tìm thấy:
/content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_val/VisDrone2019-DET-val/annotations
/content/drive/MyDrive/visdrone-small-object-detection/data/VisDrone_train/VisDrone2019-DET-train/annotations


In [17]:

from pathlib import Path
from PIL import Image
import shutil

DATA_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection/data"
)
YOLO_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection/data_yolo"
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

CLASS_NAMES = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor",
]

def find_split_roots(split):
    """Tìm thư mục annotation thuộc train hoặc val."""
    found = []

    for ann_dir in DATA_DIR.rglob("annotations"):
        if not ann_dir.is_dir():
            continue

        path_text = str(ann_dir).lower()

        if split == "train":
            valid = "train" in path_text and "test" not in path_text
        else:
            valid = "val" in path_text and "test" not in path_text

        if valid:
            found.append(ann_dir)

    return found

def find_images(folder):
    return [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

def convert_split(split):
    annotation_dirs = find_split_roots(split)

    if not annotation_dirs:
        raise FileNotFoundError(
            f"Không tìm thấy thư mục annotations cho {split}. "
            "Hãy kiểm tra kết quả ở mục 5.1."
        )

    output_images = YOLO_DIR / split / "images"
    output_labels = YOLO_DIR / split / "labels"
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    converted = 0
    missing = 0
    skipped_existing = 0

    for ann_dir in annotation_dirs:
        # Thường annotations và images nằm cùng cấp.
        image_dir = ann_dir.parent / "images"

        if not image_dir.exists():
            # Nếu cấu trúc lồng nhau, tìm thư mục images gần nhất.
            candidates = [
                p for p in ann_dir.parent.rglob("images")
                if p.is_dir()
            ]
            if not candidates:
                print("Không tìm thấy images tương ứng:", ann_dir)
                continue
            image_dir = candidates[0]

        images = find_images(image_dir)
        image_by_stem = {p.stem: p for p in images}

        for ann_file in ann_dir.glob("*.txt"):
            image_path = image_by_stem.get(ann_file.stem)

            if image_path is None:
                missing += 1
                continue

            # Giữ tên duy nhất nếu có nhiều thư mục trùng tên ảnh.
            unique_name = (
                f"{ann_dir.parent.name}_{image_path.name}"
            )
            out_image = output_images / unique_name
            out_label = output_labels / f"{Path(unique_name).stem}.txt"

            if out_image.exists() and out_label.exists():
                skipped_existing += 1
                continue

            with Image.open(image_path) as im:
                width, height = im.size

            labels = []

            for line in ann_file.read_text(
                encoding="utf-8-sig"
            ).splitlines():
                values = line.strip().split(",")

                if len(values) < 8:
                    continue

                try:
                    x, y, w, h = map(float, values[:4])
                    score = int(float(values[4]))
                    category = int(float(values[5]))
                except ValueError:
                    continue

                # Chỉ giữ 10 lớp đối tượng hợp lệ.
                if category not in range(1, 11) or score != 1:
                    continue

                x1 = max(0.0, min(x, width))
                y1 = max(0.0, min(y, height))
                x2 = max(0.0, min(x + w, width))
                y2 = max(0.0, min(y + h, height))

                bw = x2 - x1
                bh = y2 - y1

                if bw <= 0 or bh <= 0:
                    continue

                xc = ((x1 + x2) / 2) / width
                yc = ((y1 + y2) / 2) / height
                nw = bw / width
                nh = bh / height

                labels.append(
                    f"{category - 1} {xc:.6f} {yc:.6f} "
                    f"{nw:.6f} {nh:.6f}"
                )

            shutil.copy2(image_path, out_image)
            out_label.write_text(
                "\n".join(labels),
                encoding="utf-8"
            )
            converted += 1

    print(f"\n{split.upper()}:")
    print("Ảnh đã chuyển:", converted)
    print("Ảnh thiếu annotation tương ứng:", missing)
    print("Ảnh đã có từ lần chạy trước:", skipped_existing)

convert_split("train")
convert_split("val")



TRAIN:
Ảnh đã chuyển: 6471
Ảnh thiếu annotation tương ứng: 0
Ảnh đã có từ lần chạy trước: 0

VAL:
Ảnh đã chuyển: 548
Ảnh thiếu annotation tương ứng: 0
Ảnh đã có từ lần chạy trước: 0


In [18]:

import yaml

YOLO_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection/data_yolo"
)

DATA_YAML = YOLO_DIR / "visdrone.yaml"

config = {
    "path": str(YOLO_DIR),
    "train": "train/images",
    "val": "val/images",
    "names": {
        i: name for i, name in enumerate(CLASS_NAMES)
    },
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print(DATA_YAML.read_text(encoding="utf-8"))
print("Đã tạo cấu hình:", DATA_YAML)


path: /content/drive/MyDrive/visdrone-small-object-detection/data_yolo
train: train/images
val: val/images
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor

Đã tạo cấu hình: /content/drive/MyDrive/visdrone-small-object-detection/data_yolo/visdrone.yaml


In [19]:

for split in ["train", "val"]:
    image_dir = YOLO_DIR / split / "images"
    label_dir = YOLO_DIR / split / "labels"

    images = [
        p for p in image_dir.glob("*")
        if p.suffix.lower() in IMAGE_EXTS
    ]
    labels = list(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Số ảnh:", len(images))
    print("Số file nhãn:", len(labels))

    if not images or not labels:
        print("CẢNH BÁO: dữ liệu chưa đầy đủ.")



TRAIN
Số ảnh: 6471
Số file nhãn: 6471

VAL
Số ảnh: 548
Số file nhãn: 548


Bước 6: Huấn luyện mô hình YOLO baseline trên VisDrone-DET.


In [20]:

%pip install -q ultralytics pyyaml pandas matplotlib


In [21]:

import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.182
GPU khả dụng: True
GPU: Tesla T4


In [22]:

from pathlib import Path
import yaml

PROJECT_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection"
)

DATA_YAML = (
    PROJECT_DIR / "data_yolo" / "visdrone.yaml"
)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_YAML}. "
        "Hãy hoàn thành Bước 5 trước."
    )

with open(DATA_YAML, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Cấu hình:", config)

for split in ["train", "val"]:
    image_dir = Path(config["path"]) / config[split]
    count = sum(
        1 for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower()
        in {".jpg", ".jpeg", ".png", ".bmp"}
    ) if image_dir.exists() else 0

    print(f"{split}: {count} ảnh")

    if count == 0:
        raise RuntimeError(
            f"Không có ảnh trong {image_dir}. "
            "Chưa nên huấn luyện."
        )


Cấu hình: {'path': '/content/drive/MyDrive/visdrone-small-object-detection/data_yolo', 'train': 'train/images', 'val': 'val/images', 'names': {0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car', 4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle', 8: 'bus', 9: 'motor'}}
train: 6471 ảnh
val: 548 ảnh


In [23]:

from ultralytics import YOLO
from pathlib import Path
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection"
)

DATA_YAML = (
    PROJECT_DIR / "data_yolo" / "visdrone.yaml"
)

RUN_DIR = PROJECT_DIR / "runs"

# Mô hình nhỏ để giảm yêu cầu tính toán.
model = YOLO("yolo11n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=10,
    imgsz=640,
    batch=8,
    workers=2,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUN_DIR),
    name="baseline_yolo11n",
    exist_ok=True,
    seed=42,
    plots=True,
    save=True,
    patience=5
)

print("Đã kết thúc lượt huấn luyện thử.")


Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/visdrone-small-object-detection/data_yolo/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yol

In [24]:

from ultralytics import YOLO
from pathlib import Path
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/visdrone-small-object-detection"
)

BEST_MODEL = (
    PROJECT_DIR / "runs" / "baseline_yolo11n"
    / "weights" / "best.pt"
)

DATA_YAML = PROJECT_DIR / "data_yolo" / "visdrone.yaml"

if not BEST_MODEL.exists():
    raise FileNotFoundError(
        f"Không tìm thấy trọng số: {BEST_MODEL}. "
        "Kiểm tra log huấn luyện và thư mục runs."
    )

model = YOLO(str(BEST_MODEL))

metrics = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True
)

print("\n===== KẾT QUẢ BASELINE =====")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"mAP@0.5:    {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")


Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,599,238 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 70.4±34.8 MB/s, size: 142.5 KB)
val: Scanning /content/drive/MyDrive/visdrone-small-object-detection/data_yolo/val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 143.7Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=317 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.7it/s 12.1s
                   all        548      38759      0.392      0.285      0.249      0.138
            pedestrian        520       8844   

In [25]:

import pandas as pd
from pathlib import Path

RESULT_DIR = (
    Path("/content/drive/MyDrive/"
         "visdrone-small-object-detection/results")
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

baseline = pd.DataFrame([{
    "model": "YOLO11n",
    "epochs": 10,
    "image_size": 640,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
}])

csv_path = RESULT_DIR / "baseline_results.csv"

if csv_path.exists():
    old = pd.read_csv(csv_path)
    baseline = pd.concat([old, baseline], ignore_index=True)

baseline.to_csv(csv_path, index=False)

print(baseline.tail())
print("Đã lưu kết quả:", csv_path)


     model  epochs  image_size  precision   recall     mAP50  mAP50_95
0  YOLO11n      10         640    0.39176  0.28487  0.249278   0.13776
Đã lưu kết quả: /content/drive/MyDrive/visdrone-small-object-detection/results/baseline_results.csv
